# Question 19: Implement Continuous Batching for LLM Inference

### Difficulty: Hard

### Problem Statement

In naive LLM serving, a batch of requests is processed together and the entire batch must wait until the longest sequence finishes. This wastes compute on short requests that finish early.

**Continuous batching** (also called **iteration-level scheduling**) solves this by allowing the scheduler to:
- **Evict** completed sequences from the batch immediately (when they hit EOS or max length).
- **Insert** new waiting requests into the freed slots without waiting for the whole batch to finish.

This is the core scheduling idea behind systems like vLLM, TGI, and Orca.

Your task is to:
1. Implement a `Request` class to track each inference request's state.
2. Implement a `ContinuousBatchScheduler` that manages an active batch, inserting and evicting requests dynamically.
3. Show that continuous batching achieves higher throughput than naive sequential processing.

---

### Requirements

1. **`Request`** class:
   - Stores `request_id`, `input_ids`, `generated_ids`, `max_gen_len`, `is_done` flag.
   - Tracks when the request was created and when it finished.

2. **`ContinuousBatchScheduler`** class:
   - `max_batch_size`: Maximum concurrent requests.
   - `add_request(request)`: Add a request to the waiting queue.
   - `step(model)`: Run one generation step:
     a. Fill empty batch slots from the waiting queue.
     b. Run the model forward pass for all active requests.
     c. Append generated tokens to each request.
     d. Evict completed requests (EOS token or max length reached).
   - `run(model)`: Run until all requests (queued + active) are finished.

3. **Validation**: Submit 10 requests of varying lengths. Compare throughput (tokens/sec) of continuous batching vs. sequential processing.

---

### Constraints

- Use only PyTorch operations.
- Use a simple dummy model (e.g., random logit generation) so the focus is on the scheduling logic.
- Track timestamps for each request to show latency improvements.

---

### Company Tags

Perplexity, Together AI, Anyscale, Meta

---

<details>
  <summary>Hint</summary>

  - The scheduler maintains two data structures: a `waiting_queue` (list of requests not yet started) and an `active_batch` (list of requests currently being generated).
  - At each `step()`, first fill empty slots in the active batch from the waiting queue, then run inference, then evict finished requests.
  - For padding in the batch forward pass, you can either pad to the longest active sequence or process each request independently. Padding is simpler to implement.
  - The dummy model can just produce random logits -- the scheduling logic is the same regardless.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from dataclasses import dataclass, field
from typing import List, Optional

In [ ]:
# Configuration
torch.manual_seed(42)

vocab_size = 256
d_model = 128
max_batch_size = 4
eos_token_id = 0  # Token ID 0 is EOS

device = "cpu"
print(f"Vocab size: {vocab_size}, Max batch size: {max_batch_size}, EOS token: {eos_token_id}")

In [ ]:
class DummyLLM(nn.Module):
    """
    A simple dummy language model for testing the scheduler.
    Just maps embeddings through a linear layer to produce logits.
    The model itself is not the focus -- the scheduling logic is.
    """
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, input_ids):
        """input_ids: (batch_size, seq_len) -> logits: (batch_size, seq_len, vocab_size)"""
        x = self.embedding(input_ids)
        return self.fc(x)


print("DummyLLM defined.")

In [ ]:
@dataclass
class Request:
    """
    Represents a single inference request.
    
    Fields:
        request_id: Unique identifier
        input_ids: List of input token IDs (the prompt)
        max_gen_len: Maximum number of tokens to generate
        generated_ids: List of generated token IDs (grows over time)
        is_done: Whether generation is complete
        start_time: When the request started processing
        end_time: When the request finished
    """
    # TODO: Define the fields listed above
    # Use field(default_factory=list) for generated_ids
    # Use None as default for start_time and end_time
    # Use False as default for is_done
    ...
    
    def get_full_sequence(self):
        """Return the full token sequence (input + generated)."""
        # TODO
        ...
    
    def total_len(self):
        """Return the total sequence length."""
        # TODO
        ...


class ContinuousBatchScheduler:
    """
    Continuous batching scheduler that dynamically adds/removes requests.
    
    - Maintains a waiting queue and an active batch.
    - At each step, fills empty slots, runs inference, evicts finished requests.
    """
    
    def __init__(self, max_batch_size, eos_token_id, vocab_size):
        # TODO: Initialize waiting_queue, active_batch, completed list,
        #       and store max_batch_size, eos_token_id, vocab_size
        ...
    
    def add_request(self, request):
        """Add a request to the waiting queue."""
        # TODO
        ...
    
    def _fill_batch(self):
        """Move requests from waiting queue to active batch (up to max_batch_size)."""
        # TODO: While there are empty slots and waiting requests,
        #       pop from waiting queue, set start_time, add to active batch
        ...
    
    def _evict_completed(self):
        """Remove completed requests from the active batch."""
        # TODO: Move done requests to completed list, set end_time,
        #       keep only active (not done) requests
        ...
    
    def step(self, model):
        """
        Run one generation step:
        1. Fill empty batch slots from waiting queue.
        2. For each active request, run the model and sample next token.
        3. Check for completion (EOS or max length).
        4. Evict completed requests.
        
        Returns:
            bool: True if there are still active or waiting requests.
        """
        # TODO: Implement the step logic described above.
        # For simplicity, process each active request individually
        # (no padding needed).
        ...
    
    def run(self, model):
        """
        Run the scheduler until all requests are completed.
        Returns the list of completed requests.
        """
        # TODO: Call step() in a loop until no more work to do
        ...


print("Request and ContinuousBatchScheduler classes defined. Fill in the TODO sections above.")

In [ ]:
# ============================================================
# Validation: Continuous Batching vs Sequential Processing
# ============================================================

torch.manual_seed(42)
model = DummyLLM(vocab_size, d_model).to(device).eval()

# Create 10 requests with varying prompt lengths and max generation lengths
requests = []
for i in range(10):
    prompt_len = torch.randint(3, 10, (1,)).item()
    max_gen_len = torch.randint(5, 20, (1,)).item()
    input_ids = torch.randint(1, vocab_size, (prompt_len,)).tolist()  # Avoid EOS=0 in prompt
    requests.append(Request(
        request_id=i,
        input_ids=input_ids,
        max_gen_len=max_gen_len
    ))
    print(f"Request {i}: prompt_len={prompt_len}, max_gen_len={max_gen_len}")

print(f"\nTotal requests: {len(requests)}")
print(f"Max batch size: {max_batch_size}")
print()

# --- Method 1: Sequential processing ---
import copy
seq_requests = copy.deepcopy(requests)

start = time.time()
total_tokens_seq = 0
with torch.no_grad():
    for req in seq_requests:
        req.start_time = time.time()
        current_ids = torch.tensor([req.input_ids], device=device)
        for _ in range(req.max_gen_len):
            logits = model(current_ids)
            next_logits = logits[:, -1, :]
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            req.generated_ids.append(next_token)
            if next_token == eos_token_id:
                break
            current_ids = torch.cat([current_ids, torch.tensor([[next_token]], device=device)], dim=1)
        req.end_time = time.time()
        total_tokens_seq += len(req.generated_ids)
seq_time = time.time() - start

# --- Method 2: Continuous batching ---
cb_requests = copy.deepcopy(requests)
scheduler = ContinuousBatchScheduler(max_batch_size, eos_token_id, vocab_size)
for req in cb_requests:
    scheduler.add_request(req)

start = time.time()
with torch.no_grad():
    completed = scheduler.run(model)
cb_time = time.time() - start

total_tokens_cb = sum(len(r.generated_ids) for r in completed)

print("=" * 50)
print("Results")
print("=" * 50)
print(f"\nSequential: {seq_time:.4f}s, {total_tokens_seq} tokens, {total_tokens_seq/seq_time:.1f} tokens/sec")
print(f"Continuous: {cb_time:.4f}s, {total_tokens_cb} tokens, {total_tokens_cb/cb_time:.1f} tokens/sec")
print(f"\nSpeedup: {seq_time / cb_time:.2f}x")
print()

# Show per-request latency for continuous batching
print("Per-request completion times (continuous batching):")
for r in sorted(completed, key=lambda x: x.request_id):
    latency = r.end_time - r.start_time
    print(f"  Request {r.request_id}: generated {len(r.generated_ids)} tokens, latency={latency:.4f}s")